# Brain MRI Tumor Contouring — Train on Colab

End-to-end notebook: install deps, load the dataset from Google Drive, train the U-Net, plot curves, visualize predicted contours, and export the trained weights.

**Before running:** Runtime -> Change runtime type -> **GPU** (T4 is plenty).

You need two things:
1. This repo on GitHub (set `REPO_URL` below).
2. The dataset `archive.zip` uploaded to the root of your Google Drive.

## 1. Clone the repo and install dependencies

In [ ]:
# Your repo:
REPO_URL = "https://github.com/shibdad/brain-mri-unet.git"

!git clone $REPO_URL
%cd brain-mri-unet
!pip install -q -r requirements.txt

## 2. Get the dataset from your Google Drive

Upload `archive.zip` (the dataset you downloaded from Kaggle) to the **root of your Google Drive** ("My Drive"). The cell below mounts Drive and unzips it — no Kaggle token needed.

*(If you'd rather pull it straight from Kaggle instead, see the commented alternative in the cell.)*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Unzip the dataset from your Drive into data/. Adjust the path if your zip
# has a different name or location in My Drive.
ZIP_PATH = '/content/drive/MyDrive/archive.zip'

!mkdir -p data/lgg-mri-segmentation
!unzip -q -o "$ZIP_PATH" -d data/lgg-mri-segmentation
!echo "Extracted. Mask files found:" && find data -name "*_mask.tif" | wc -l

# --- Alternative: download directly from Kaggle instead of Drive ---
# from google.colab import files
# files.upload()  # select kaggle.json
# import os; os.makedirs('/root/.kaggle', exist_ok=True)
# !cp kaggle.json /root/.kaggle/kaggle.json && chmod 600 /root/.kaggle/kaggle.json
# !python scripts/download_data.py --out data

## 3. Inspect a few image / mask pairs

In [ ]:
import matplotlib.pyplot as plt
from src.data import find_samples, split_by_patient
import cv2, numpy as np

samples = find_samples('data/lgg-mri-segmentation')
train, val, test = split_by_patient(samples)
print(f'total slices: {len(samples)} | train {len(train)} val {len(val)} test {len(test)}')
print(f'patients -> train/val/test split is patient-level (no leakage)')

# Show a few slices that actually contain tumor.
tumor = [s for s in samples if cv2.imread(s.mask_path, 0).max() > 0][:4]
fig, ax = plt.subplots(2, len(tumor), figsize=(4*len(tumor), 8))
for i, s in enumerate(tumor):
    img = cv2.cvtColor(cv2.imread(s.image_path), cv2.COLOR_BGR2RGB)
    msk = cv2.imread(s.mask_path, 0)
    ax[0, i].imshow(img); ax[0, i].set_title('FLAIR slice'); ax[0, i].axis('off')
    ax[1, i].imshow(msk, cmap='gray'); ax[1, i].set_title('tumor mask'); ax[1, i].axis('off')
plt.tight_layout(); plt.show()

## 4. Train

~50 epochs on a T4 takes roughly 30-60 min. Lower `--epochs` for a quick test.

In [ ]:
!python -m src.train --data-dir data/lgg-mri-segmentation \
    --epochs 50 --batch-size 32 --lr 1e-4 --out runs/exp1

## 5. Plot the training curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv('runs/exp1/metrics.csv')
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(df.epoch, df.train_loss, label='train'); ax[0].plot(df.epoch, df.val_loss, label='val')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(df.epoch, df.train_dice, label='train'); ax[1].plot(df.epoch, df.val_dice, label='val')
ax[1].set_title('Dice'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.savefig('assets/training_curves.png', dpi=120); plt.show()
print('best val dice:', df.val_dice.max())

## 6. Visualize predicted contours on held-out test slices

In [ ]:
import torch, cv2, numpy as np, matplotlib.pyplot as plt
from src.inference import load_model, predict_mask
from src.utils import overlay_mask

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = load_model('runs/exp1/best_model.pt', device)

tumor_test = [s for s in test if cv2.imread(s.mask_path, 0).max() > 0][:4]
fig, ax = plt.subplots(1, len(tumor_test), figsize=(5*len(tumor_test), 5))
for i, s in enumerate(tumor_test):
    img = cv2.cvtColor(cv2.imread(s.image_path), cv2.COLOR_BGR2RGB)
    gt = cv2.imread(s.mask_path, 0)
    pred = predict_mask(model, img, device)
    ov = overlay_mask(img, gt, color=(255,0,0))      # ground truth in red
    ov = overlay_mask(ov, pred, color=(0,255,0))     # prediction in green
    ax[i].imshow(ov); ax[i].axis('off'); ax[i].set_title('GT (red) vs Pred (green)')
plt.tight_layout(); plt.savefig('assets/sample_predictions.png', dpi=120); plt.show()

## 7. Save the weights

Download the checkpoint, or copy it to Google Drive so you can reuse it for the Gradio demo.

In [ ]:
from google.colab import files
files.download('runs/exp1/best_model.pt')

# --- or save to Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# !cp runs/exp1/best_model.pt /content/drive/MyDrive/